# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [22]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier, RandomForestRegressor, RandomForestClassifier, AdaBoostRegressor, AdaBoostClassifier, GradientBoostingRegressor, GradientBoostingClassifier

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score, classification_report, confusion_matrix

from sklearn.model_selection import GridSearchCV


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
spaceship = spaceship.dropna()

In [6]:
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship["Cabin"].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

In [7]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

In [8]:
features = spaceship.drop(columns=["Transported"])
target = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state = 17)

In [9]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output = False)

train_encoded = encoder.fit_transform(X_train[["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]])

train_encoded_df = pd.DataFrame(train_encoded, columns=encoder.get_feature_names_out(), index = X_train.index)

test_encoded = encoder.transform(X_test[["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]])

test_encoded_df = pd.DataFrame(test_encoded, columns=encoder.get_feature_names_out(), index = X_test.index)



In [10]:
numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

X_train_final = pd.concat([X_train[numerical_cols], train_encoded_df], axis=1)

X_test_final = pd.concat([X_test[numerical_cols], test_encoded_df], axis=1)

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

scaler = StandardScaler()
normalizer = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

In [18]:
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=20), n_estimators=100, max_samples=1000)

bagging_clf.fit(X_train_scaled, y_train)

pred = bagging_clf.predict(X_test_scaled)

print("Accuracy: ", accuracy_score(y_test, pred))
print("Score: ", bagging_clf.score(X_test_scaled, y_test))
print("Confusion Matrix: ", confusion_matrix(y_test, pred))

Accuracy:  0.8086232980332829
Score:  0.8086232980332829
Confusion Matrix:  [[516 114]
 [139 553]]


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

- Evaluate your model

In [1]:
#your code here

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [19]:
grid = {"n_estimators": [50, 100,200], 
        "estimator__max_leaf_nodes": [250,500],
        "estimator__max_depth": [10,30]}

In [20]:
bagging = BaggingClassifier(DecisionTreeClassifier(), random_state=17)

- Run Grid Search

In [23]:
grid_search = GridSearchCV(
    estimator=bagging,
    param_grid=grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

- Evaluate your model

In [24]:
grid_search.fit(X_train_scaled, y_train)

print(grid_search.best_params_)

print(grid_search.best_score_)

best_model = grid_search.best_estimator_

pred = best_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))


{'estimator__max_depth': 30, 'estimator__max_leaf_nodes': 250, 'n_estimators': 200}
0.7920101559588314
Accuracy: 0.8071104387291982
Confusion Matrix:
[[510 120]
 [135 557]]


The hyperparameters did not improve our model, it actually got worse, since the new accuracy is 0.807, while before we had 0.808.